In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [2]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [3]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = """
Please solve the following task:

{test_case["task"]}
    """
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output
    
def run_test_case(test_case):
    """Runs the test case and returns the result"""
    output = run_prompt(test_case)
    # TODO - Grading
    score = 10

    return {
        "output": output,
        "score": score, 
        "test_case": test_case
    }

def run_eval(dataset):
    """Runs the eval on the dataset and returns the results"""
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    return results

In [4]:
import json

from prompt_toolkit import prompt


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []


    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    # 0.0 No creativity, 1.0 Maximum creativity.
    answer = chat(messages, stop_sequences = ["```"])
    return json.loads(answer)

In [5]:
dataset = generate_dataset()
#dataset
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [7]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)
print(json.dumps(results, indent=2)) 

[
  {
    "output": "I'd be happy to help solve the task! However, I notice that the task content isn't showing up in your message. The `{test_case[\"task\"]}` appears to be a template variable that wasn't filled in.\n\nCould you please provide:\n1. The actual task description or problem you'd like me to solve\n2. Any relevant context, data, or examples\n3. The expected output format\n\nOnce you share those details, I'll do my best to help you solve it!",
    "score": 10,
    "test_case": {
      "task": "Write a Python function that validates if a string is a valid AWS S3 bucket name. S3 bucket names must be between 3 and 63 characters long, contain only lowercase letters, numbers, hyphens, and periods, and cannot start or end with a hyphen or period."
    }
  },
  {
    "output": "I'd be happy to help solve a task, but I don't see the actual task content in your message. The reference to `{test_case[\"task\"]}` appears to be a template variable that wasn't filled in.\n\nCould you ple